# Visualize Gate MLPs (R¹→R¹)

In [1]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
from jaxtyping import Float, Int

from spd.models.component_model import ComponentModel, Config
from spd.models.component_utils import calc_component_acts, calc_masks
from spd.models.components import (
    AnyComponent,
    AnyGate,
)
from spd.plotting import plot_mask_vals

torch.set_grad_enabled(False)
# Set your model path here
# TODO: make this more general
# MODEL_PATH = "spd/experiments/tms/out/randrecon1.00e+00_p1.00e+00_lpsp1.00e-04_m200_sd0_lr1.00e-03_bs4096_ft40_hid10hid-layers1_20250609_151342_824/model_40000.pth" # noqa: E501
MODEL_PATH = "../spd/experiments/tms/out/randrecon1.00e+00_p2.00e+00_lpsp1.00e-04_m50_sd0_lr1.00e-03_bs4096_ft40_hid10hid-layers1_20250617_195956_677/model_10000.pth"  # noqa: E501

DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# Load the decomposition model
model: ComponentModel
config: Config
model, config, _ = ComponentModel.from_pretrained(MODEL_PATH)

# Extract components and gates for plotting
components: dict[str, AnyComponent] = {
    k: v for k, v in model.components.items() if hasattr(v, "A") and hasattr(v, "B")
}  # type: ignore
gates: dict[str, AnyGate] = {k: v for k, v in model.gates.items()}

# Get the input shape from your TMS config
n_features: int = 40
model = model.to(DEVICE)

FileNotFoundError: [Errno 2] No such file or directory: '../spd/experiments/tms/out/randrecon1.00e+00_p2.00e+00_lpsp1.00e-04_m50_sd0_lr1.00e-03_bs4096_ft40_hid10hid-layers1_20250617_195956_677/model_10000.pth'

In [ ]:
# Generate the mask plots with single feature inputs (one-hot vectors)
figures, perm_indices = plot_mask_vals(
    model=model,
    components=components,
    gates=gates,
    batch_shape=(n_features,),
    device=DEVICE,
    input_magnitude=1.0,
    plot_regular_masks=False,  # Only plot sparsity masks (red plots)
)

In [ ]:
top_k_dead_alive: int = 40  # Number of top alive and dead components
alive_indicies: Int[torch.Tensor, "top_k_dead_alive"] = perm_indices["linear1"][:top_k_dead_alive]
dead_indicies: Int[torch.Tensor, "top_k_dead_alive"] = perm_indices["linear1"][top_k_dead_alive:]

In [ ]:
# Create magnitude levels
n_magnitudes: int = 100
magnitudes: Float[torch.Tensor, "n_magnitudes"] = torch.linspace(0, 1, n_magnitudes, device=DEVICE)

# Create batch: each one-hot vector at each magnitude
batch: Float[torch.Tensor, "n_features*n_magnitudes n_features"] = torch.zeros(
    n_features * n_magnitudes,
    n_features,
    device=DEVICE,
)

for i, magnitude in enumerate(magnitudes):
    start_idx = i * n_features
    end_idx = (i + 1) * n_features
    batch[start_idx:end_idx] = torch.eye(n_features, device=DEVICE) * magnitude

plt.matshow(batch.cpu().numpy()[:500].T, cmap="gray")
plt.matshow(batch.cpu().numpy()[-500:].T, cmap="gray")

In [ ]:
# Calculate component activations (A matrices applied to pre-weight acts)
pre_gate_acts = model.forward_with_pre_forward_cache_hooks(
    batch, module_names=list(model.gates.keys())
)[1]

target_component_acts = calc_component_acts(
    pre_weight_acts=pre_gate_acts,
    As={module_name: v.A for module_name, v in components.items()},
)  # "input into MLP"

# Calculate gate outputs (post-gate activations)
masks, sparsity_masks = calc_masks(
    gates=gates,
    target_component_acts=target_component_acts,
    detach_inputs=False,
)

In [ ]:
# Get raw MLP outputs (before final nonlinearity)
raw_mlp_outputs: dict = {}
for gate_name, gate in gates.items():
    gate_input = target_component_acts[gate_name]
    raw_mlp_outputs[gate_name] = gate._compute_pre_activation(gate_input)

In [ ]:
def plot_component_io_scatter(
    *,
    target_component_acts: dict[str, Float[torch.Tensor, "*batch n_components"]],
    raw_mlp_outputs: dict[str, Float[torch.Tensor, "*batch n_components"]],
    submodule: str,
    component_indices: Int[torch.Tensor, " n_plot"],
    cols: int = 5,
    ensure_zero: bool = False,
    title: str | None = None,
    figsize: tuple[int, int] | None = None,
    show: bool = True,
) -> tuple[plt.Figure, np.ndarray]:
    """Scatter-plot MLP pre-activations against raw MLP outputs.

    Plots each selected component of a given submodule in its own axis.
    The function reproduces the four repetitive plotting cells at the
    end of the notebook.

    # Parameters:
     - `target_component_acts : dict[str, Float[torch.Tensor, "*batch n_components"]]`
       Mapping from submodule name to *pre-gate* activations (A @ input).
     - `raw_mlp_outputs : dict[str, Float[torch.Tensor, "*batch n_components"]]`
       Mapping from submodule name to raw MLP outputs (before non-linearity).
     - `submodule : str`
       Name of the submodule to plot (e.g. `"linear1"` or `"linear2"`).
     - `component_indices : Int[torch.Tensor, "n_plot"]`
       1-D tensor containing component indices to scatter-plot.
     - `cols : int`
       Number of subplot columns (defaults to `5`).
     - `ensure_zero : bool`
       Expand the y-axis to include `0` (use `True` for *dead* components).
     - `title : str | None`
       Figure suptitle. If `None`, a sensible default is generated.
     - `figsize : tuple[int, int] | None`
       Size passed to `plt.subplots`; defaults to `(15, 3*rows)`.
     - `show : bool`
       Call `plt.show()` before returning.

    # Returns:
     - `tuple[plt.Figure, np.ndarray]`
       The created `(figure, flat_axes)` pair.

    # Usage:

    ```python
    fig, axes = plot_component_io_scatter(
        target_component_acts=target_component_acts,
        raw_mlp_outputs=raw_mlp_outputs,
        submodule="linear1",
        component_indices=alive_indicies[:10],
        ensure_zero=False,
        title="TMS 40-10, linear1 MLP input-outputs. Alive Components",
    )
    ```
    """
    n_components: int = int(component_indices.numel())
    rows: int = int(math.ceil(n_components / cols))
    figsize = figsize or (15, 3 * rows)

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows == 1:
        axes = axes.reshape(1, -1)
    flat_axes: np.ndarray = axes.flatten()

    x_mat: Float[torch.Tensor, "*batch n_components"] = target_component_acts[submodule]
    y_mat: Float[torch.Tensor, "*batch n_components"] = raw_mlp_outputs[submodule]

    for ax_idx, comp_idx_tensor in enumerate(component_indices):
        comp_idx: int = int(comp_idx_tensor.item())
        x = x_mat[:, comp_idx].cpu()
        y = y_mat[:, comp_idx].cpu()

        flat_axes[ax_idx].scatter(x, y, s=1, c="black", alpha=0.1)
        flat_axes[ax_idx].set_title(f"Component {comp_idx}")
        flat_axes[ax_idx].grid(True, alpha=0.3)
        flat_axes[ax_idx].set_xlim(-2.0, 2.0)

        if ensure_zero:
            y_min, y_max = flat_axes[ax_idx].get_ylim()
            eps: float = 0.01
            flat_axes[ax_idx].set_ylim(min(y_min, 0.0) - eps, max(y_max, 0.0) + eps)

        flat_axes[ax_idx].set_xlabel("Pre-activation")
        flat_axes[ax_idx].set_ylabel("Raw MLP Output")

    # Hide any unused subplot axes.
    for idx in range(n_components, len(flat_axes)):
        flat_axes[idx].set_visible(False)

    fig.suptitle(
        title or f"TMS 40-10, {submodule} MLP input-outputs.",
        fontsize=16,
    )
    plt.tight_layout()

    if show:
        plt.show()

    return fig, flat_axes

In [ ]:
_ = plot_component_io_scatter(
    target_component_acts=target_component_acts,
    raw_mlp_outputs=raw_mlp_outputs,
    submodule="linear1",
    component_indices=alive_indicies[:10],
    ensure_zero=False,
    title="TMS 40-10, linear1 MLP input-outputs. Alive Components",
)

_ = plot_component_io_scatter(
    target_component_acts=target_component_acts,
    raw_mlp_outputs=raw_mlp_outputs,
    submodule="linear1",
    component_indices=dead_indicies[:10],
    ensure_zero=True,
    title="TMS 40-10, linear1 MLP input-outputs. Dead Components",
)

_ = plot_component_io_scatter(
    target_component_acts=target_component_acts,
    raw_mlp_outputs=raw_mlp_outputs,
    submodule="linear2",
    component_indices=alive_indicies[:10],
    ensure_zero=False,
    title="TMS 40-10, linear2 MLP input-outputs. Alive Components",
)

_ = plot_component_io_scatter(
    target_component_acts=target_component_acts,
    raw_mlp_outputs=raw_mlp_outputs,
    submodule="linear2",
    component_indices=dead_indicies[:10],
    ensure_zero=True,
    title="TMS 40-10, linear2 MLP input-outputs. Dead Components",
)